# M3 step 7: image quality with SSIM and LPIPS (task 3)

PSNR was already measured in notebook 4 (31.50 dB). The proposal also asks for:

- **SSIM**, which compares structure. Higher is better; 1 means identical.
  `skimage.metrics.structural_similarity`, on the full-size RGB photos.
- **LPIPS**, which compares what people actually notice. Lower is better; 0 means identical.
  The `lpips` package with the AlexNet version.

Each of the 500 original photos is compared with its stamped PNG, at full size. PSNR is measured again
on the same pairs, so all three numbers in the table come from the same photos.

**GPU:** not needed. About 5 minutes on a CPU (LPIPS downloads AlexNet weights the first time).

## Setup

**Kaggle:** turn on *Settings > Accelerator > GPU T4 x2* (only notebooks 8, 9 and 11 need it) and
*Internet on*, and attach the `veristamp-m3-inputs` dataset (README.md, "Running on Kaggle"). The first
cell then finds the code, installs what is missing and moves into `m3_experiments`. No path to edit.

**Colab or a local machine:** set the paths once in `settings.py`, then run from the `m3_experiments`
folder.

In [ ]:
# Kaggle setup: find the dataset wherever Kaggle put it, copy the code, point the notebooks at the files.
import os, shutil, subprocess, sys
from pathlib import Path

def find(rel, max_depth=6):
    for depth in range(max_depth + 1):
        hits = sorted(Path('/kaggle/input').glob('/'.join(['*'] * depth + [rel])))
        if hits:
            return hits[0]
    return None

if Path('/kaggle/input').is_dir() and not Path('settings.py').exists():
    code = find('m3_experiments/settings.py')
    model = find('m3/best.pt')
    print('Code found at: ', code)
    print('Model found at:', model)
    if code is None or model is None:
        print(subprocess.run(['find', '/kaggle/input', '-maxdepth', '4'], capture_output=True, text=True).stdout[:3000])
        raise SystemExit('The veristamp-m3-inputs dataset is not attached, or has a different layout (see the list above).')
    repo = Path('/kaggle/working/VeriStamp')
    if not (repo / 'm3_experiments' / 'settings.py').exists():
        shutil.rmtree(repo, ignore_errors=True)          # removes a failed earlier clone
        shutil.copytree(code.parent.parent, repo, ignore=shutil.ignore_patterns('.git'))
    # Each item is found on its own, so the inputs can come in several zips or datasets.
    paths = {'M3E_MODEL_PATH': 'm3/best.pt', 'M3E_TEST_DIR': 'm3/test', 'M3E_TEST_MANIFEST': 'm3/test_manifest.csv',
             'M3E_GENIMAGE_SPLIT_DIR': 'genimage/splits', 'M3E_GENIMAGE_ROOT': 'genimage/images',
             'M3E_M1_CHECKPOINT': 'm1/best.pt', 'M3E_M1_VSLIB_DIR': 'm1/vslib_m1_final'}
    for name, rel in paths.items():
        hit = find(rel)
        os.environ[name] = str(hit if hit else model.parent.parent / rel)   # the notebooks run from here inherit these
        print(f'  {rel:<22} {"found" if hit else "not in the inputs"}')
    os.environ['M3E_WORK_DIR'] = '/kaggle/working/m3_work'
    os.chdir(repo / 'm3_experiments')
    for pkg in ('diffusers', 'lpips'):
        try:
            __import__(pkg)
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
print('Working in', Path.cwd())

In [ ]:
# On Colab instead, uncomment and run once per session:
# from google.colab import drive; drive.mount('/content/drive')
# !git clone https://github.com/KruthikaVirupakshappa/VeriStamp.git /content/VeriStamp
# %cd /content/VeriStamp/m3_experiments
# !pip install -q diffusers transformers accelerate scikit-image lpips

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt

import settings as S
import m3_eval as E
mw = S.setup()                      # imports ../m3_watermark/m3_watermark.py as it is
torch.manual_seed(S.SEED)
np.random.seed(S.SEED)

m3 = mw.load_m3(S.MODEL_PATH)       # load once, reuse for every photo
DEVICE = m3.device
print('Model epoch', m3.epoch, 'on', DEVICE)

In [ ]:
files = E.list_photos(S.TEST_DIR, S.N_TEST_PHOTOS)
if len(files) != S.N_TEST_PHOTOS:
    raise RuntimeError(f'Expected {S.N_TEST_PHOTOS} test photos in {S.TEST_DIR}, found {len(files)}')
# Every photo gets a fixed verdict, P(AI) and tag (E.payload_for), the same in every notebook.
# The stamped PNGs are made once and reused by notebooks 6 to 9.
records = E.stamp_test_set(mw, m3, files, S.WORK_DIR, S.MODEL_PATH, S.SEED)
print(len(records), 'stamped photos in', S.WORK_DIR / 'stamped')
records.head(3)

## Measure every photo

In [ ]:
lp = E.load_lpips(DEVICE)
rows = []
for _, r in E.tqdm(records.iterrows(), total=len(records)):
    a, b = E.open_rgb(r['original']), E.open_rgb(r['stamped'])
    rows.append({'name': r['name'], 'width': r['width'], 'height': r['height'], 'psnr': E.psnr(a, b),
                 'ssim': E.ssim(a, b), 'lpips': E.lpips_distance(lp, a, b, DEVICE)})
quality = pd.DataFrame(rows)
summary = pd.DataFrame({'metric': ['PSNR (dB)', 'SSIM', 'LPIPS (AlexNet)'],
                        'mean': [quality.psnr.mean(), quality.ssim.mean(), quality.lpips.mean()],
                        'std': [quality.psnr.std(), quality.ssim.std(), quality.lpips.std()],
                        'median': [quality.psnr.median(), quality.ssim.median(), quality.lpips.median()],
                        'worst': [quality.psnr.min(), quality.ssim.min(), quality.lpips.max()]})
summary

## The photos where the stamp is most visible

The guide notes the stamp can be faintly visible in smooth areas such as sky. The photos with the
highest LPIPS show where.

In [ ]:
worst = quality.sort_values('lpips', ascending=False).head(3)
fig, axes = plt.subplots(len(worst), 3, figsize=(15, 4.5 * len(worst)), squeeze=False)
for row_axes, (_, w) in zip(axes, worst.iterrows()):
    r = records.set_index('name').loc[w['name']]
    a, b = E.open_rgb(r['original']), E.open_rgb(r['stamped'])
    diff = np.clip(np.abs(np.asarray(b, np.float32) - np.asarray(a, np.float32)) * 10, 0, 255).astype(np.uint8)
    for ax, im, t in zip(row_axes, [a, b, diff], ['Original', f"Stamped (LPIPS {w['lpips']:.3f})", 'Difference x10']):
        ax.imshow(im); ax.set_title(t); ax.axis('off')
plt.tight_layout()
(S.RESULTS_DIR / 'figures').mkdir(exist_ok=True)
plt.savefig(S.RESULTS_DIR / 'figures' / 'image_quality_worst.png', dpi=100, bbox_inches='tight')
plt.show()

## Save

In [ ]:
quality.to_csv(S.RESULTS_DIR / 'image_quality_per_photo.csv', index=False)
summary.to_csv(S.RESULTS_DIR / 'image_quality_summary.csv', index=False)
(S.RESULTS_DIR / 'image_quality_summary.md').write_text(E.to_markdown(summary) + '\n')
E.save_results(S.RESULTS_DIR, 'image_quality', meta={'n_photos': len(quality), 'lpips_net': 'alex',
               'ssim': 'skimage structural_similarity, channel_axis=2, data_range=255, full size',
               'environment': E.environment_info()})
print(E.to_markdown(summary))
if abs(quality.psnr.mean() - 31.50) > 0.05:
    print('PSNR differs from notebook 4 (31.50 dB): check the setup.')